In [2]:
from pathlib import Path
import sys
sys.executable
import os
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import pearsonr, spearmanr

REPO_ROOT = Path.cwd().parents[1]  # falls Notebook in notebooks/... liegt
DATA_PATH = REPO_ROOT / "data" / "all_heuristics_dataset.pkl"

OUT_DIR = REPO_ROOT / "data" / "outputs" / "ppp_global"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PPP_TEMPLATE_PATH = REPO_ROOT / "src" / "prompt_templates" / "ppp_with_refs.md"

print("REPO_ROOT:", REPO_ROOT)
print("DATA_PATH exists:", DATA_PATH.exists(), DATA_PATH)
print("OUT_DIR:", OUT_DIR)
print("PPP_TEMPLATE_PATH exists:", PPP_TEMPLATE_PATH.exists(), PPP_TEMPLATE_PATH)

# make repo importable
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("sys.path[0]:", sys.path[0])


REPO_ROOT: /Users/emirhangunes/VSCode/ppp-performance-prediction
DATA_PATH exists: True /Users/emirhangunes/VSCode/ppp-performance-prediction/data/all_heuristics_dataset.pkl
OUT_DIR: /Users/emirhangunes/VSCode/ppp-performance-prediction/data/outputs/ppp_global
PPP_TEMPLATE_PATH exists: True /Users/emirhangunes/VSCode/ppp-performance-prediction/src/prompt_templates/ppp_with_refs.md
sys.path[0]: /Users/emirhangunes/VSCode/ppp-performance-prediction


In [4]:
import os
from src.api.deepseek_config import load_deepseek_config
from src.api.deepseek_client import DeepSeekLLMClient


cfg = load_deepseek_config()
client = DeepSeekLLMClient(cfg)

# Preflight (Auth + basic response)
resp = client.generate('Return JSON only: {"ping":"pong"}', temperature=0.0, max_tokens=30)
print(resp.text[:120])


```json
{"ping":"pong"}
```


In [5]:
df = pd.read_pickle(DATA_PATH)

required = {"heuristic_id", "raw_app_type", "code", "objective"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.dropna(subset=["code", "objective"]).copy()
df["heuristic_id"] = df["heuristic_id"].astype(str)
df["raw_app_type"] = df["raw_app_type"].astype(str)
df["objective"] = pd.to_numeric(df["objective"], errors="coerce")
df = df.dropna(subset=["objective"]).reset_index(drop=True)

print("Usable rows:", len(df))
df.head(3)


Usable rows: 15507


,heuristic_id,raw_app_type,instance_scale,filename,strategy,algorithm,code,objective,task_name,parse_ok,is_timeout
0,pop_0_op_e1_n0_251224_134701,bin_greedy,results_20251224_134317_bp_online_qwen25coder3...,program_pop_0_op_e1_n0_251224_134701.json,e1,The new algorithm assigns scores based on a co...,"import numpy as np\n\ndef score(item, bins):\n...",1.51534,BinPacking,True,False
1,pop_0_op_e1_n10_251224_134701,bin_greedy,results_20251224_134317_bp_online_qwen25coder3...,program_pop_0_op_e1_n10_251224_134701.json,e1,The new algorithm calculates scores for each b...,"import numpy as np\n\ndef score(item, bins):\n...",0.32770,BinPacking,True,False
2,pop_0_op_e1_n11_251224_134701,bin_greedy,results_20251224_134317_bp_online_qwen25coder3...,program_pop_0_op_e1_n11_251224_134701.json,e1,The new algorithm calculates scores for each b...,"import numpy as np\n\ndef score(item, bins):\n...",1.51534,BinPacking,True,False


In [6]:
task_bounds = (
    df.groupby("raw_app_type")["objective"]
      .agg(["min", "max"])
      .to_dict(orient="index")
)
task_bounds


{'bin_greedy': {'min': 0.00563, 'max': 1.51534},
 'cvrp_lns': {'min': 3343425.3, 'max': 3981765.6},
 'premarshalling_astar': {'min': 0.08148, 'max': 13.9329},
 'puzzle_astar': {'min': 0.4574, 'max': 3.65888}}

In [7]:
import numpy as np

def select_refs_stratified(df_task: pd.DataFrame, target_id: str, k: int = 7):
    df_cand = (
        df_task[df_task["heuristic_id"] != target_id]
        .sort_values("objective", ascending=True)
        .reset_index(drop=True)
    )
    n = len(df_cand)
    if n < k:
        return None

    # evenly spaced quantiles incl. ends
    idxs = np.linspace(0, n - 1, k).round().astype(int).tolist()

    refs = []
    seen = set()
    for i in idxs:
        row = df_cand.iloc[int(i)]
        hid = str(row["heuristic_id"])
        if hid not in seen:
            refs.append(row)
            seen.add(hid)

    # safety: fill if duplicates collapsed
    if len(refs) < k:
        for _, row in df_cand.iterrows():
            hid = str(row["heuristic_id"])
            if hid not in seen:
                refs.append(row)
                seen.add(hid)
            if len(refs) >= k:
                break

    return refs[:k]



def build_refs_block(refs):
    lines = []
    for i, r in enumerate(refs, 1):
        lines += [
            f"{i})",
            "Heuristic code:",
            "```python",
            str(r["code"]),
            "```",
            f"Objective: {float(r['objective'])}",
            ""
        ]
    return "\n".join(lines).strip() + "\n"


In [8]:
ppp_template = PPP_TEMPLATE_PATH.read_text(encoding="utf-8")

def render_ppp_prompt(*, raw_app_type, task_min, task_max, references_block, target_code):
    return ppp_template.format(
        raw_app_type=raw_app_type,
        task_min=task_min,
        task_max=task_max,
        references_block=references_block,
        target_code=target_code,
    )

print("Template loaded, length:", len(ppp_template))


Template loaded, length: 752


In [9]:
from src.api.parse import parse_ppp_response


In [13]:
TEST_PER_TASK = 3

for app, df_task in df.groupby("raw_app_type"):
    task_min = task_bounds[app]["min"]
    task_max = task_bounds[app]["max"]

    print("\n=== APP:", app, "| rows:", len(df_task), "| range:", (task_min, task_max))
    df_small = df_task.head(TEST_PER_TASK)

    for _, row in df_small.iterrows():
        hid = row["heuristic_id"]
        refs = select_refs_stratified(df_task, hid, k=5)
        if refs is None:
            print("No refs for", hid)
            continue

        prompt = render_ppp_prompt(
            raw_app_type=app,
            task_min=task_min,
            task_max=task_max,
            references_block=build_refs_block(refs),
            target_code=str(row["code"]),
        )

        resp = client.generate(prompt, temperature=0.2, max_tokens=512)
        pred, conf, ok, err, _ = parse_ppp_response(resp.text)

        if pred is not None:
            pred = max(task_min, min(task_max, pred))  # clamp to bounds

        print("hid:", hid, "| obj:", row["objective"], "| pred:", pred, "| conf:", conf, "| ok:", ok, "| err:", err)



=== APP: bin_greedy | rows: 4445 | range: (0.00563, 1.51534)
hid: pop_0_op_e1_n0_251224_134701 | obj: 1.51534 | pred: 0.03501 | conf: 0.65 | ok: True | err: None
hid: pop_0_op_e1_n10_251224_134701 | obj: 0.3277 | pred: 0.02832 | conf: 0.65 | ok: True | err: None
hid: pop_0_op_e1_n11_251224_134701 | obj: 1.51534 | pred: 0.045 | conf: 0.65 | ok: True | err: None

=== APP: cvrp_lns | rows: 1557 | range: (3343425.3, 3981765.6)
hid: pop_0_op_e1_n0_251225_153652 | obj: 3978416.0 | pred: 3650000.0 | conf: 0.65 | ok: True | err: None
hid: pop_0_op_e1_n11_251225_153652 | obj: 3613762.5 | pred: 3710000.0 | conf: 0.65 | ok: True | err: None
hid: pop_0_op_e1_n12_251225_153652 | obj: 3956702.6 | pred: 3670000.0 | conf: 0.65 | ok: True | err: None

=== APP: premarshalling_astar | rows: 4647 | range: (0.08148, 13.9329)
hid: pop_0_op_e1_n0_250815_114538 | obj: 10.69957 | pred: 0.210055 | conf: 0.65 | ok: True | err: None
hid: pop_0_op_e1_n10_250815_114538 | obj: 8.79957 | pred: 0.57161 | conf: 0.75 |

In [ ]:
RUN_TAG = "deepseek_k5"  # später z.B. "ollama_k5" / "llama_k5"
PPP_PKL = OUT_DIR / f"ppp_results_{RUN_TAG}.pkl"
PPP_CSV = OUT_DIR / f"ppp_results_{RUN_TAG}.csv"

if PPP_PKL.exists():
    prev = pd.read_pickle(PPP_PKL)
    prev["heuristic_id"] = prev["heuristic_id"].astype(str)
    done_ids = set(prev["heuristic_id"])  # wir resumieren auf Basis: "row existiert"
    results = prev.to_dict(orient="records")
    print("Resuming, loaded:", len(done_ids))
else:
    done_ids = set()
    results = []

t0 = time.time()

for app, df_task in df.groupby("raw_app_type"):
    task_min = task_bounds[app]["min"]
    task_max = task_bounds[app]["max"]

    for _, row in tqdm(df_task.iterrows(), total=len(df_task), desc=f"PPP [{app}]"):
        hid = str(row["heuristic_id"])
        if hid in done_ids:
            continue

        refs = select_refs_stratified(df_task, hid, k=5)
        if refs is None:
            # not enough refs; record a row so resume doesn't keep retrying forever
            results.append({
                "heuristic_id": hid,
                "raw_app_type": app,
                "objective": float(row["objective"]),
                "prediction": None,
                "confidence": None,
                "parse_ok": False,
                "parse_error": "Not enough reference heuristics for k=5",
            })
            done_ids.add(hid)
            continue

        prompt = render_ppp_prompt(
            raw_app_type=app,
            task_min=task_min,
            task_max=task_max,
            references_block=build_refs_block(refs),
            target_code=str(row["code"]),
        )

        try:
            resp = client.generate(
                prompt,
                temperature=0.2,
                max_tokens=512,
                meta={"stage": "ppp", "heuristic_id": hid, "raw_app_type": app},
            )

            pred, conf, ok, err, _ = parse_ppp_response(resp.text)

            if pred is not None:
                pred = max(task_min, min(task_max, pred))  # clamp

            results.append({
                "heuristic_id": hid,
                "raw_app_type": app,
                "objective": float(row["objective"]),
                "prediction": pred,
                "confidence": conf,
                "parse_ok": bool(ok),
                "parse_error": err,
            })

        except Exception as e:
            # record failure, continue
            results.append({
                "heuristic_id": hid,
                "raw_app_type": app,
                "objective": float(row["objective"]),
                "prediction": None,
                "confidence": None,
                "parse_ok": False,
                "parse_error": f"LLM call failed: {type(e).__name__}: {e}",
            })

        done_ids.add(hid)

        if len(results) % 200 == 0:
            tmp = pd.DataFrame(results)
            tmp.to_pickle(PPP_PKL)
            tmp.to_csv(PPP_CSV, index=False)
            print("[checkpoint] saved", len(results))

elapsed = time.time() - t0
ppp_df = pd.DataFrame(results)
ppp_df.to_pickle(PPP_PKL)
ppp_df.to_csv(PPP_CSV, index=False)

print("DONE. Rows:", len(ppp_df), "Time(s):", round(elapsed, 1))
print("Saved:", PPP_PKL)
print("Saved:", PPP_CSV)
ppp_df.head()


PPP [bin_greedy]:   4%|▍         | 200/4445 [06:04<2:08:45,  1.82s/it]

[checkpoint] saved 200


PPP [bin_greedy]:   9%|▉         | 400/4445 [12:32<1:51:06,  1.65s/it]

[checkpoint] saved 400


PPP [bin_greedy]:  13%|█▎        | 600/4445 [19:02<1:57:41,  1.84s/it]

...

PPP [puzzle_astar]:  73%|███████▎  | 3551/4858 [1:44:52<36:25,  1.67s/it]

[checkpoint] saved 14200


PPP [puzzle_astar]:  77%|███████▋  | 3751/4858 [1:50:29<28:53,  1.57s/it]

[checkpoint] saved 14400


PPP [puzzle_astar]:  79%|███████▉  | 3845/4858 [1:53:06<30:02,  1.78s/it]

In [12]:
import pandas as pd

PPP_CSV = OUT_DIR / "ppp_results_deepseek_k5.csv"  # ggf. anpassen!
prev = pd.read_csv(PPP_CSV)
prev["heuristic_id"] = prev["heuristic_id"].astype(str)

done_ids = set(prev["heuristic_id"])
results = prev.to_dict(orient="records")

print("Resuming from CSV. Done:", len(done_ids))


Resuming from CSV. Done: 14400


In [13]:
import pandas as pd

PPP_CSV = OUT_DIR / "ppp_results_deepseek_k5.csv"   # <-- falls dein CSV anders heißt: anpassen!

prev = pd.read_csv(PPP_CSV)
prev["heuristic_id"] = prev["heuristic_id"].astype(str)

done_ids = set(prev["heuristic_id"])
results = prev.to_dict(orient="records")

print("Resume from CSV. Rows loaded:", len(prev))
print("Unique IDs done:", len(done_ids))
print(prev["raw_app_type"].value_counts())


Resume from CSV. Rows loaded: 14400
Unique IDs done: 14400
raw_app_type
premarshalling_astar    4647
bin_greedy              4445
puzzle_astar            3751
cvrp_lns                1557
Name: count, dtype: int64


In [18]:
from scipy.stats import pearsonr, spearmanr

def topk_overlap(pred, true, ids, k=10):
    """
    pred, true: 1D arrays
    ids: heuristic_id array
    """
    assert len(pred) == len(true) == len(ids)

    # sort indices (lower is better)
    idx_pred = sorted(range(len(pred)), key=lambda i: pred[i])
    idx_true = sorted(range(len(true)), key=lambda i: true[i])

    top_pred = set(ids[i] for i in idx_pred[:k])
    top_true = set(ids[i] for i in idx_true[:k])

    return len(top_pred & top_true) / k


In [ ]:
import pandas as pd
import numpy as np

K_TOP = 10   # <--- hier variabel ändern (5, 10, 20 …)

results_eval = []

for app, df_task in ppp_df.groupby("raw_app_type"):
    df_task = df_task.dropna(subset=["prediction", "objective"])
    if len(df_task) < K_TOP:
        continue

    pred = df_task["prediction"].values
    true = df_task["objective"].values
    ids  = df_task["heuristic_id"].astype(str).values

    pearson_r, _  = pearsonr(pred, true)
    spearman_r, _ = spearmanr(pred, true)
    overlap_k     = topk_overlap(pred, true, ids, k=K_TOP)

    results_eval.append({
        "task": app,
        "n": len(df_task),
        "pearson": pearson_r,
        "spearman": spearman_r,
        f"top_{K_TOP}_overlap": overlap_k,
    })

eval_df = pd.DataFrame(results_eval)
eval_df

,task,n,pearson,spearman,top_10_overlap
0,bin_greedy,4444,0.238527,0.437158,0.5
1,cvrp_lns,1052,0.514450,0.411831,0.0
2,premarshalling_astar,1653,0.011219,0.219386,0.0
3,puzzle_astar,4858,0.366392,0.323773,0.0


In [33]:
# --- Repro-Setup ---
RUN_TAG_REP = "deepseek_k5_repeats"
PPP_REP_PKL = OUT_DIR / f"ppp_repeats_{RUN_TAG_REP}.pkl"
PPP_REP_CSV = OUT_DIR / f"ppp_repeats_{RUN_TAG_REP}.csv"

N_PER_TASK = 10     # 10–20 empfohlen
N_REPEATS  = 10     # 10–20 empfohlen
K_REFS     = 5      # muss zu deinem Template/Setup passen
TEMP       = 0.2
MAX_TOKENS = 512

print("Will run:", {"N_PER_TASK": N_PER_TASK, "N_REPEATS": N_REPEATS, "K_REFS": K_REFS})
print("Output:", PPP_REP_CSV)


Will run: {'N_PER_TASK': 10, 'N_REPEATS': 10, 'K_REFS': 5}
Output: /Users/emirhangunes/VSCode/ppp-performance-prediction/data/outputs/ppp_global/ppp_repeats_deepseek_k5_repeats.csv


In [34]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

selected = []
for app, df_task in df.groupby("raw_app_type"):
    # nur tasks nehmen, die du wirklich willst; sonst kommentier das raus
    # if app not in ["CVRP-LNS", "Premarshalling-A*", "Puzzle-A*", "BinGreedy"]:
    #     continue

    task_min = task_bounds[app]["min"]
    task_max = task_bounds[app]["max"]

    # Heuristiken mischen und prüfen, ob refs existieren
    ids = df_task["heuristic_id"].astype(str).unique().tolist()
    rng.shuffle(ids)

    ok_ids = []
    for hid in ids:
        refs = select_refs_stratified(df_task, hid, k=K_REFS)
        if refs is not None:
            ok_ids.append(hid)
        if len(ok_ids) >= N_PER_TASK:
            break

    print(app, "| candidates:", len(ids), "| usable:", len(ok_ids))
    selected += [{"raw_app_type": app, "heuristic_id": hid, "task_min": task_min, "task_max": task_max} for hid in ok_ids]

sel_df = pd.DataFrame(selected)
display(sel_df.head())
print("Total selected:", len(sel_df))


bin_greedy | candidates: 4444 | usable: 10
cvrp_lns | candidates: 1052 | usable: 10
premarshalling_astar | candidates: 1653 | usable: 10
puzzle_astar | candidates: 4858 | usable: 10


,raw_app_type,heuristic_id,task_min,task_max
0,bin_greedy,pop_7_op_e1_n12_251224_212213,0.00563,1.51534
1,bin_greedy,pop_5_op_e2_n0_251225_033249,0.00563,1.51534
2,bin_greedy,pop_10_op_e1_n11_251224_163329,0.00563,1.51534
3,bin_greedy,pop_8_op_e1_n2_251224_160509,0.00563,1.51534
4,bin_greedy,pop_1_op_e1_n2_251224_135754,0.00563,1.51534


Total selected: 40


In [ ]:
import time
from tqdm import tqdm

# Resume laden (falls abgebrochen)
if PPP_REP_PKL.exists():
    prev_rep = pd.read_pickle(PPP_REP_PKL)
    results_rep = prev_rep.to_dict(orient="records")
    done_keys = set(zip(prev_rep["raw_app_type"].astype(str), prev_rep["heuristic_id"].astype(str), prev_rep["repeat_idx"].astype(int)))
    print("Resuming repeats:", len(done_keys), "done")
else:
    results_rep = []
    done_keys = set()

t0 = time.time()

# Für schnellen Zugriff: df pro task
task_to_df = {app: d.copy() for app, d in df.groupby("raw_app_type")}

for _, srow in tqdm(sel_df.iterrows(), total=len(sel_df), desc="Tasks"):
    app = str(srow["raw_app_type"])
    hid = str(srow["heuristic_id"])
    task_min = float(srow["task_min"])
    task_max = float(srow["task_max"])

    df_task = task_to_df[app]
    row = df_task[df_task["heuristic_id"].astype(str) == hid].iloc[0]

    # References einmal bauen (gleich halten über repeats -> fairer Stabilitätstest)
    refs = select_refs_stratified(df_task, hid, k=K_REFS)
    if refs is None:
        # sollte durch Auswahl eigentlich nicht passieren
        continue

    prompt = render_ppp_prompt(
        raw_app_type=app,
        task_min=task_min,
        task_max=task_max,
        references_block=build_refs_block(refs),
        target_code=str(row["target_code"]),
    )

    for r in range(N_REPEATS):
        key = (app, hid, r)
        if key in done_keys:
            continue

        try:
            resp = client.generate(
                prompt,
                temperature=TEMP,
                max_tokens=MAX_TOKENS,
                meta={"stage": "ppp_repeats", "heuristic_id": hid, "raw_app_type": app, "repeat_idx": r},
            )
            pred, conf, ok, err, _ = parse_ppp_response(resp.text)

            if pred is not None:
                pred = max(task_min, min(task_max, pred))

            results_rep.append({
                "raw_app_type": app,
                "heuristic_id": hid,
                "repeat_idx": r,
                "objective": float(row["objective"]),
                "prediction": pred,
                "confidence": conf,
                "parse_ok": bool(ok),
                "parse_error": err,
            })

        except Exception as e:
            results_rep.append({
                "raw_app_type": app,
                "heuristic_id": hid,
                "repeat_idx": r,
                "objective": float(row["objective"]),
                "prediction": None,
                "confidence": None,
                "parse_ok": False,
                "parse_error": f"LLM call failed: {type(e).__name__}: {e}",
            })

        done_keys.add(key)

        # checkpoint
        if len(results_rep) % 200 == 0:
            tmp = pd.DataFrame(results_rep)
            tmp.to_pickle(PPP_REP_PKL)
            tmp.to_csv(PPP_REP_CSV, index=False)
            print("[checkpoint] saved", len(results_rep))

rep_df = pd.DataFrame(results_rep)
rep_df.to_pickle(PPP_REP_PKL)
rep_df.to_csv(PPP_REP_CSV, index=False)

print("DONE. Rows:", len(rep_df), "Time(s):", round(time.time() - t0, 1))
print("Saved:", PPP_REP_CSV)
rep_df.head()


Tasks:   0%|          | 0/40 [00:00<?, ?it/s]


KeyError: 'code'